# Dataset Information

Dataset: Annual net earnings (earn_nt_net).

Source: Eurostat.

License: Eurostat reuse policy.

Access method: Eurostat API via the eurostat Python package.

Link: https://ec.europa.eu/eurostat/databrowser/view/earn_nt_net/default/table?lang=en


In [1]:
## Installing eurostat library in case it doesn't exist
#%pip install eurostat

## Load dataset

In [2]:
import eurostat

df = eurostat.get_data_df("earn_nt_net")

In [3]:
## Saving raw dataset

In [4]:
df.to_csv(
    "../data/raw/eurostat_wages.csv",
    index = False
)

In [5]:
## Data check and transformation

In [6]:
df.shape

(8892, 31)

In [7]:
df.head()

,freq,currency,estruct,ecase,geo\TIME_PERIOD,2000,2001,2002,2003,2004,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,EUR,FAM,CPL_CH2_AW100,AT,4338.57,4338.57,4341.60,4516.80,4516.80,...,5098.40,5098.40,5168.00,5168.00,5888.00,5168.00,7178.00,5992.60,5336.10,5581.40
1,A,EUR,FAM,CPL_CH2_AW100,BE,2881.91,2857.10,2933.19,2937.96,2996.64,...,3800.73,3876.75,3933.08,3985.78,4087.20,4312.56,4394.78,4725.70,5034.59,5135.41
2,A,EUR,FAM,CPL_CH2_AW100,BG,NaN,NaN,NaN,NaN,NaN,...,521.52,521.52,552.20,552.20,552.20,552.20,674.91,539.92,828.31,828.31
3,A,EUR,FAM,CPL_CH2_AW100,CH,3210.73,3377.69,3511.93,3420.98,3398.11,...,5503.58,5397.14,5194.81,5393.74,5604.86,5549.90,5971.93,6174.11,NaN,NaN
4,A,EUR,FAM,CPL_CH2_AW100,CY,NaN,NaN,NaN,NaN,NaN,...,NaN,1040.00,1040.00,1055.70,1060.49,1060.49,1065.48,1182.27,1225.54,1310.04


In [8]:
df.columns.to_list()

['freq',
 'currency',
 'estruct',
 'ecase',
 'geo\\TIME_PERIOD',
 '2000',
 '2001',
 '2002',
 '2003',
 '2004',
 '2005',
 '2006',
 '2007',
 '2008',
 '2009',
 '2010',
 '2011',
 '2012',
 '2013',
 '2014',
 '2015',
 '2016',
 '2017',
 '2018',
 '2019',
 '2020',
 '2021',
 '2022',
 '2023',
 '2024',
 '2025']

### Reading description of columns from the Eurostat links:
ecase - earnings case. Data is splitted for single person without children with earnigs from 50 up to 167 percent of average, one-earner couple, two-earner couple with childrens etc. For the analysis most reliable variable seems to be single person without children earning 100% of the average ("P1_NCH_AW100"), as it is the most intuitive model (typical employee with annual net income) for calculating affordability index.

estruct - earnings structure. Net earning ("NET") is the most reliable variable for the analysis.

geo - geopolitical entity. To keep consistency there are two variables that will be taken: "EU27_2020", and "PL".

time - whole dataset contains data from 2000 until 2025. For the analysis 2007 until 2024 will be taken.

freq - time frequency. There is only annual variable.

## Data filtration

In [9]:
currency_filter = df["currency"].isin(["EUR"])
estruct_filter = df["estruct"].isin(["NET"])
ecase_filter = df["ecase"].isin(["P1_NCH_AW100"])
geo_filter = df["geo\\TIME_PERIOD"].isin(["PL", "EU27_2020"])

In [10]:
filtered_df = df[
currency_filter &
estruct_filter &
ecase_filter &
geo_filter
]

In [11]:
filtered_df.head()

,freq,currency,estruct,ecase,geo\TIME_PERIOD,2000,2001,2002,2003,2004,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
1335,A,EUR,NET,P1_NCH_AW100,EU27_2020,NaN,NaN,NaN,NaN,NaN,...,22239.91,22597.24,23166.71,23849.60,24047.22,25104.11,26653.72,28323.92,28776.80,29953.59
1351,A,EUR,NET,P1_NCH_AW100,PL,4153.75,4882.23,4750.55,4338.05,4289.22,...,8200.43,8975.57,9618.59,10225.78,11376.35,11134.57,12503.72,14487.18,17339.93,19086.30


## Changing wide format into long format

In [12]:
period_columns = []

for col in filtered_df.columns:
    if col.isdigit() and 2007 <= int(col) <= 2024:
        period_columns.append(col)

print(period_columns)
    

['2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


In [13]:
melted_df = filtered_df.melt(
    id_vars = ["geo\\TIME_PERIOD"],
    value_vars = period_columns,
    var_name = "period",
    value_name = "annual net earnings"
)

In [14]:
melted_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   geo\TIME_PERIOD      36 non-null     object 
 1   period               36 non-null     object 
 2   annual net earnings  30 non-null     float64
dtypes: float64(1), object(2)
memory usage: 992.0+ bytes


### Corecting data types and renaming columns names

In [15]:
melted_df = melted_df.rename(
    columns = {
        "geo\\TIME_PERIOD":"country",
        "period":"year"
    }
)

melted_df["year"] = melted_df["year"].astype(int)

In [16]:
melted_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   country              36 non-null     object 
 1   year                 36 non-null     int64  
 2   annual net earnings  30 non-null     float64
dtypes: float64(1), int64(1), object(1)
memory usage: 992.0+ bytes


### Checking nulls

In [17]:
melted_df.isna().sum()

country                0
year                   0
annual net earnings    6
dtype: int64

In [18]:
melted_df[melted_df["annual net earnings"].isna()]

,country,year,annual net earnings
0,EU27_2020,2007,NaN
2,EU27_2020,2008,NaN
4,EU27_2020,2009,NaN
6,EU27_2020,2010,NaN
8,EU27_2020,2011,NaN
10,EU27_2020,2012,NaN


### There are nulls for EU27 between 2007-2012. Checking whether these values "are hidden" in different geo\\TIME_PERIOD code

In [19]:
df["geo\\TIME_PERIOD"].unique()

array(['AT', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EA19', 'EA20',
       'EA21', 'EE', 'EL', 'ES', 'EU15', 'EU27_2020', 'EU28', 'FI', 'FR',
       'HR', 'HU', 'IE', 'IS', 'IT', 'JP', 'LT', 'LU', 'LV', 'MT', 'NL',
       'NO', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK', 'TR', 'UK', 'US'],
      dtype=object)

In [20]:
eu_filter = df["geo\\TIME_PERIOD"].isin([
    "EU15",
    "EU27_2020",
    "EU28"
])

eu_df = df[
    currency_filter &
    estruct_filter &
    ecase_filter &
    eu_filter
]

eu_df
    

,freq,currency,estruct,ecase,geo\TIME_PERIOD,2000,2001,2002,2003,2004,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
1334,A,EUR,NET,P1_NCH_AW100,EU15,20081.39,20805.28,21378.66,21383.09,22215.57,...,27881.09,27916.66,28469.24,29289.75,NaN,NaN,NaN,NaN,NaN,NaN
1335,A,EUR,NET,P1_NCH_AW100,EU27_2020,NaN,NaN,NaN,NaN,NaN,...,22239.91,22597.24,23166.71,23849.60,24047.22,25104.11,26653.72,28323.92,28776.8,29953.59
1336,A,EUR,NET,P1_NCH_AW100,EU28,NaN,NaN,NaN,NaN,NaN,...,23997.96,24148.05,24732.16,25519.70,NaN,NaN,NaN,NaN,NaN,NaN


### To keep consistency between input data in this and other notebooks EU27 will be kept. 
### This means the affordabiliy analysis will cover years 2013-2024, because of lack of data for prior years for EU27. 

In [21]:
melted_df = melted_df.dropna()

In [22]:
melted_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30 entries, 1 to 35
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   country              30 non-null     object 
 1   year                 30 non-null     int64  
 2   annual net earnings  30 non-null     float64
dtypes: float64(1), int64(1), object(1)
memory usage: 960.0+ bytes


In [23]:
melted_df.groupby("country")["year"].agg(["min", "max", "count"])

,min,max,count
country,,,
EU27_2020,2013,2024,12
PL,2007,2024,18


In [24]:
melted_df = melted_df[
    melted_df["year"] >= 2013
]

In [25]:
melted_df.groupby("country")["year"].agg(["min", "max", "count"])

,min,max,count
country,,,
EU27_2020,2013,2024,12
PL,2013,2024,12


In [26]:
melted_df.isna().sum()

country                0
year                   0
annual net earnings    0
dtype: int64

### Renaming country names "EU27_2020" to "European Union (27)" and "PL" to "Poland" to fit with other data tables (electricity_price, population, co2)

In [31]:
melted_df["country"] = melted_df["country"].replace({"EU27_2020":"European Union (27)",
                                                    "PL":"Poland"})

In [32]:
melted_df.head()

,country,year,annual net earnings
12,European Union (27),2013,21323.68
13,Poland,2013,7466.23
14,European Union (27),2014,21608.28
15,Poland,2014,7991.17
16,European Union (27),2015,21953.76


## Saving the output

In [33]:
melted_df.to_csv("../data/processed/eurostat_wages_processed.csv", 
                 index = False
                )